This notebook is the nodes extraction note book this will be a manual approach but will be further automated

Mongo db ping

In [1]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


Open AI ping

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4.5-preview', 'gpt-4.5-preview-2025-02-27', 'gpt-4o-mini-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-audio-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4o-mini-realtime-preview', 'o1-mini-2024-09-12', 'o1-mini', 'omni-moderation-latest', 'gpt-4o-mini-audio-preview', 'omni-moderation-2024-09-26', 'whisper-1', 'gpt-4o-realtime-preview-2024-10-01', 'babbage-002', 'gpt-4-turbo-preview', 'chatgpt-4o-latest', 'tts-1-hd-1106', 'text-embedding-3-large', 'gpt-4-0125-preview', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4', 'gpt-4o-2024-05-13', 'tts-1-hd', 'o1-preview', 'o1-preview-2024-09-12', 'gpt-4o-2024-11-20', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini', 'tts-1', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4-turbo', 'gpt-3.5-turbo-instruct', 'o1', 'gpt-4o-2024-08-06', 'gpt-3.5-turbo-0125', 'gpt-4o-realtime-preview-2024-12-17', 'gpt

Get the first five articles from mongo db

In [4]:
import json
from bson import json_util
def fetch_articles(limit=10):
    try:
        # Fetch articles with an optional limit
        articles_cursor = articles_collection.find().limit(limit)
        articles = list(articles_cursor)

        if articles:
            print(f"✅ Retrieved {len(articles)} articles from MongoDB.\n")
            for idx, article in enumerate(articles, start=1):
                print(f"--- Article {idx} ---")
                # Convert BSON to JSON using json_util
                article_json = json.dumps(article, indent=4, default=json_util.default)
                print(article_json)
        else:
            print("⚠️ No articles found in the collection.")

        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []


# Fetch and print articles in JSON format
articles = fetch_articles(limit=5)

✅ Retrieved 5 articles from MongoDB.

--- Article 1 ---
{
    "_id": {
        "$oid": "67b1e41e0554959fd725edf6"
    },
    "title": "Mitsubishi Chemical expands production capacities",
    "paragraphs": [
        {
            "p1": "Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.",
            "p2": "The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.",
            "p3": "Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com",
            "p4": "Your email address will not be published.Required fields are marked*",
          

Read prompts from file only

In [5]:
def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

Combine the articles

In [6]:
def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()


Fetch a single function

In [7]:
def fetch_single_article(skip=0):
    try:
        article = articles_collection.find().skip(skip).limit(1)
        article = list(article)

        if article:
            article_json = json.dumps(article[0], indent=4, default=json_util.default)
            print(f"✅ Retrieved Article {skip + 1}:\n{article_json}\n")
            return article[0]
        else:
            print("⚠️ No more articles found in the collection.")
            return None

    except Exception as e:
        print(f"❌ Error fetching article: {e}")
        return None

Extracting nodes from articles

In [9]:
PROMPT_PATH = "initial_node_extraction.txt"
def extract_nodes_test(text, PROMPT_PATH):

    prompt = read_prompt_from_file_only(PROMPT_PATH)

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": f'''
            Here is the article: {text}
            '''
            }
        ],
        temperature=0)

    # Ensure JSON is properly loaded
    extracted_nodes = json.loads(completion.choices[0].message.content)
    print(f"✅ Retrieved nodes {extracted_nodes}\n")
    # Extract the list of nodes correctly
    if "nodes" in extracted_nodes:
        return extracted_nodes["nodes"]
    else:
        raise ValueError("Expected 'nodes' key in LLM output but not found.")

Processing function

In [12]:
import json

def extract_nodes(article_id, text, PROMPT_PATH):
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            temperature=0
        )

        # Parse the JSON response
        extracted_data = json.loads(completion.choices[0].message.content)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("Expected 'nodes' key in LLM output but not found.")

        nodes = extracted_data["nodes"]

        # Format nodes for MongoDB insertion
        formatted_nodes = []
        for node in nodes:
            location_data = node.get("location", {})  # Get location or default to empty dict
            formatted_location = {
                "city": location_data.get("city", ""),  # Ensure city is always a string
                "country": location_data.get("country", "")  # Ensure country is always a string
            }

            formatted_nodes.append({
                "article_id": article_id,  # Reference to the source article
                "id": node.get("id"),  # Unique ID of the node
                "type": node.get("type"),  # Type of the entity (company, factory, etc.)
                "name": node.get("name"),  # Name of the entity
                "parent_company": node.get("parent_company"),  # Parent company if applicable
                "subsidiaries": node.get("subsidiaries", []),  # Ensure it's always a list
                "location": formatted_location,  # Fixed location with city & country
                "products": node.get("products", []),  # Ensure it's always a list
            })

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []


In [15]:
def process_articles(limit, offset, PROMPT_PATH):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find({}, {"_id": 1, "paragraphs": 1})  # Fetch paragraphs instead of text
        .sort("_id", 1)  # Sort by MongoDB ObjectId (ascending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)   # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH)

            # Ensure extracted nodes are valid before insertion
            if not formatted_nodes:
                print(f"⚠️ No nodes extracted for Article ID: {articleID}. Skipping update.")
                continue

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, "nodes": formatted_nodes}}  # Add new fields
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

PROCESS

In [18]:
PROMPT_PATH = "initial_node_extraction.txt"
#
process_articles(20,5, PROMPT_PATH)

🔹 Processing 20 articles (Offset: 5)
📌 Processing Article ID: 67b1e4200554959fd725edfb
✅ Retrieved nodes {'nodes': [{'id': 'northvolt', 'type': 'company', 'name': 'Northvolt', 'parent_company': None, 'subsidiaries': []}, {'id': 'scania', 'type': 'company', 'name': 'Scania', 'parent_company': None, 'subsidiaries': []}, {'id': 'abb', 'type': 'company', 'name': 'ABB', 'parent_company': None, 'subsidiaries': []}, {'id': 'factory_vasteras', 'type': 'factory', 'name': 'Northvolt Labs', 'location': {'city': 'Västerås', 'country': 'Sweden'}, 'products': ['battery cells']}, {'id': 'factory_skelleftea', 'type': 'factory', 'name': 'Northvolt Gigafactory', 'location': {'city': 'Skellefteå', 'country': 'Sweden'}, 'products': ['lithium-ion batteries']}]}

⚠️ No updates made for Article ID: 67b1e4200554959fd725edfb.
📌 Processing Article ID: 67b1e4210554959fd725edfc
✅ Retrieved nodes {'nodes': [{'id': 'tesla', 'type': 'company', 'name': 'Tesla', 'parent_company': None, 'subsidiaries': []}, {'id': 'sqm

 This additional scripts are experimental to see if we can use distributed compute to help us with the workload

In [23]:
import os
from tkinter import Tk, filedialog
import shutil

# Get the directory of the running notebook
notebook_dir = os.getcwd()  # Current working directory (where the notebook is)

# Define keystore paths in the same directory as the notebook
keystore_path = os.path.join(notebook_dir, "id.keystore")
default_keystore_path = os.path.join(notebook_dir, "default.keystore")

# Check if the keystore already exists
if not os.path.exists(keystore_path):
    print("Please select your DCP keystore file...\n")

    # Open file dialog to select the keystore file
    root = Tk()
    root.withdraw()  # Hide the root window
    root.attributes('-topmost', True)  # Bring dialog to front
    file_path = filedialog.askopenfilename(title="Select DCP Keystore File")

    if file_path:
        shutil.copy(file_path, keystore_path)  # Copy keystore file
        shutil.copy(file_path, default_keystore_path)  # Copy as default.keystore

        print(f"Keystore file saved to {notebook_dir}")
    else:
        print("No file selected. Exiting.")
else:
    print("Keystore files found in the notebook directory. Skipping upload.")

# Install DCP if not installed
try:
    import dcp
except ImportError:
    print("Installing DCP package...")
    !pip3 install dcp -q

Keystore files found in the notebook directory. Skipping upload.


In [26]:
import dcp
dcp.init()

SpiderMonkeyError: Error in file /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/urllib/request.py, on line 1322, column 1:
Error: Python URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028)>
Traceback (most recent call last):
File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/dcp/js/node_modules/dcp-client/index.py", line 44, in fetch
File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/dcp/js/node_modules/dcp-client/index.py", line 42, in fetch
File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/urllib/request.py", line 189, in urlopen
File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/urllib/request.py", line 489, in open
File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/urllib/request.py", line 506, in _open
File "/Library/Frameworks/Python.frame

In [15]:
from bson import json_util

def fetch_articles(batch_size=10, total_limit=10, offset=0):
    try:
        total_articles = articles_collection.count_documents({})
        if total_articles == 0:
            print("⚠️ No articles found in the collection.")
            return []

        total_limit = min(total_limit, total_articles - offset)  # Ensure we don't exceed available articles
        if total_limit <= 0:
            print("⚠️ No articles available after the specified offset.")
            return []

        print(f"✅ Fetching {total_limit} articles with offset {offset}.")

        articles_cursor = (
            articles_collection.find()
            .skip(offset)
            .limit(total_limit)
        )
        articles = list(articles_cursor)

        print(f"🔹 Retrieved {len(articles)} articles.\n")
        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []


In [16]:
article_batches = list(fetch_articles(batch_size=5, total_limit=10,offset=0))
print(len(article_batches))

✅ Fetching 10 articles with offset 0.
🔹 Retrieved 10 articles.

10


In [17]:
PROMPT_PATH = "initial_node_extraction.txt"
with open(PROMPT_PATH, "r", encoding="utf-8") as file:
    prompt_text = file.read()
print(prompt_text)

You are an expert energy economist who is creating a knowledge graph on clean technology factories.
Your task is to extract only the key entities (nodes) from the provided text, ensuring that each node type follows its specific schema.

**RULES FOR EXTRACTION**
1. Extract only explicitly mentioned entities in the article. Do not infer missing details.
2. Extract all mentioned nodes of each type.
3. Each entity must belong to one of the following three categories:
	•	Company (any mention of a company)
	•	Joint Venture (any mention of a joint venture)
	•	Factory (any mention of a clean technology factory: could be closed down, operating or planned for the future)
4. Ensure each entity has a unique identifier.
5. If a detail is missing, return "null" instead of inferring.
6. Output must be in JSON format using the schemas below.

**Expected JSON Output Schema**

1. Company Node Schema
{
  "id": "string",  // A unique identifier for the company (e.g., "tesla", "northvolt").
  "type": "comp

In [20]:
def work_function(batch, prompt_text, openai_api_key, mongo_url):
    dcp.progress(0)
    import pymongo
    from openai import OpenAI


    client = OpenAI(api_key=openai_api_key)
    # MongoDB Connection using the provided URL
    mongo_client = pymongo.MongoClient(mongo_url)
    db = mongo_client["your_database"]
    nodes_collection = db["nodes"]

    def combine_paragraphs(article):
        """Combines paragraphs from an article dictionary into a single string."""
        paragraphs = article.get('paragraphs', [])
        if not paragraphs:
            print(f"⚠️ No paragraphs found in article ID: {article.get('_id', 'Unknown')}.")
            return ""

        combined_text = ""
        for para_obj in paragraphs:
            for key in sorted(para_obj.keys()):  # Ensure order consistency
                combined_text += para_obj[key].strip() + " "

        return combined_text.strip()

    def extract_nodes(article_id, text, prompt_text):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": prompt_text},  # Use the passed prompt text
                    {"role": "user", "content": f"Here is the article: {text}"}
                ],
                temperature=0
            )
            extracted_data = json.loads(completion.choices[0].message.content)
            if "nodes" not in extracted_data:
                raise ValueError("Expected 'nodes' key in LLM output but not found.")

            formatted_nodes = []
            for node in extracted_data["nodes"]:
                formatted_nodes.append({
                    "article_id": article_id,
                    "id": node.get("id"),
                    "type": node.get("type"),
                    "name": node.get("name"),
                    "parent_company": node.get("parent_company"),
                    "subsidiaries": node.get("subsidiaries", []),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    },
                    "products": node.get("products", [])
                })
            return formatted_nodes
        except json.JSONDecodeError as e:
            print(f"❌ JSON Parsing Error: {e}")
            return []
        except Exception as e:
            print(f"❌ Error extracting nodes: {e}")
            return []

    for article in batch:
        article_id = str(article["_id"])
        text = combine_paragraphs(article)  # Now available inside the worker
        if not text:
            print(f"⚠️ No valid text for Article ID: {article_id}. Skipping.")
            continue

        formatted_nodes = extract_nodes(article_id, text, prompt_text)  # Use the passed prompt
        if formatted_nodes:
            nodes_collection.insert_many(formatted_nodes)
            print(f"✅ Completed Article ID: {article_id}. Inserted {len(formatted_nodes)} nodes.")

    return len(batch)# Return number of processed documents

load env

In [21]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URL = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

article batch

In [22]:
job = dcp.compute_for(article_batches, work_function, [prompt_text, OPENAI_API_KEY, MONGO_URL])
job.computeGroups = [{"joinKey": "qhala", "joinSecret": "dcp"}]
job.modules = ["pymongo", "openai"]
job.public.name = "MongoDB Data Cleaning with DCP"

import json
job.on("accepted", lambda accepted: print(f"DCP job id: {job.id}\nAccepted, awaiting results..."))
job.on("result", lambda result: print(f"    ✓ Processed batch {int(result.sliceNumber)}"))
job.on("error", lambda error: print(json.dumps(error, indent=4).replace("\\n", "\n")))

print("\nStarting data extraction and cleaning on DCP...")
job.exec()
num_processed = job.wait()

print("Total Articles Processed:", sum(num_processed))

AttributeError: module 'dcp' has no attribute 'compute_for'